<a href="https://colab.research.google.com/github/MuhammadSahlanHabibi/Implementasi-Seminar-Carrier-and-Competition-in-Industrial-era-with-GIS/blob/main/Analisis_Cakupan_Sinyal_(Coverage_Area)_dan_Deteksi_Blind_Spot_Jaringan_Wi_Fi_ODP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ==============================================================================
# IMPLEMENTASI BARU SEMINAR 3: TELCO SPATIAL DATA SCIENCE
# Topik: Analisis Coverage Area & Deteksi Blind Spot Jaringan Wi-Fi Telkomsel
# ==============================================================================

import geopandas as gpd
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap
from shapely.geometry import Point

print("Memulai Analisis Jaringan Telekomunikasi...\n")

# ==============================================================================
# 1. GENERASI DATA (Tiang ODP Telkomsel & Rumah Warga)
# ==============================================================================
np.random.seed(10)

# A. Simulasi 5 Titik Tiang ODP (Optical Distribution Point) / Router Induk
odp_lat = np.random.uniform(-3.8100, -3.7800, 5)
odp_lon = np.random.uniform(102.2600, 102.2900, 5)
gdf_odp = gpd.GeoDataFrame(
    {'ID_ODP': [f'ODP-{i+1}' for i in range(5)]},
    geometry=gpd.points_from_xy(odp_lon, odp_lat),
    crs="EPSG:4326"
)

# B. Simulasi 300 Titik Pemukiman (Pelanggan / Calon Pelanggan)
# B. Simulasi 300 Titik Pemukiman (Digeser sepenuhnya ke daratan sebelah timur)
rumah_lat = np.random.uniform(-3.8200, -3.7600, 300)
rumah_lon = np.random.uniform(102.2750, 102.3100, 300) # Geser ke kanan (menjauhi laut)
gdf_rumah = gpd.GeoDataFrame(
    {'ID_Rumah': [f'RMH-{i+1}' for i in range(300)]},
    geometry=gpd.points_from_xy(rumah_lon, rumah_lat),
    crs="EPSG:4326"
)

# ==============================================================================
# 2. SPATIAL GEOPROCESSING (COVERAGE & BLIND SPOT DETECTION)
# ==============================================================================
# Konversi ke UTM (EPSG:32748) agar satuan menjadi Meter
gdf_odp_utm = gdf_odp.to_crs(epsg=32748)
gdf_rumah_utm = gdf_rumah.to_crs(epsg=32748)

# Asumsi: Sinyal Wi-Fi / Tarikan Kabel optimal maksimal berjarak 400 Meter
# Kita buat poligon coverage gabungan dari seluruh tiang (Dissolve)
coverage_polygons = gdf_odp_utm.geometry.buffer(400)
# Gabungkan semua lingkaran menjadi 1 area sinyal raksasa
coverage_area = coverage_polygons.unary_union
gdf_coverage = gpd.GeoDataFrame(geometry=[coverage_area], crs="EPSG:32748")

# KLASIFIKASI PELANGGAN: Terjangkau vs Blind Spot
# Mengecek apakah titik rumah berada "di dalam" (within) poligon coverage
gdf_rumah_utm['Status_Sinyal'] = gdf_rumah_utm.geometry.apply(lambda x: 'Terjangkau' if x.within(coverage_area) else 'Blind Spot')

# Kembalikan ke koordinat Web/GPS
gdf_coverage_gps = gdf_coverage.to_crs(epsg=4326)
gdf_rumah_gps = gdf_rumah_utm.to_crs(epsg=4326)

# ==============================================================================
# 3. ANALISIS DATASTATISTIK
# ==============================================================================
statistik = gdf_rumah_gps['Status_Sinyal'].value_counts()
print("=== HASIL ANALISIS CAKUPAN JARINGAN (Radius 400m) ===")
print(f"Total Rumah Terjangkau Sinyal : {statistik.get('Terjangkau', 0)} rumah")
print(f"Total Rumah BLIND SPOT        : {statistik.get('Blind Spot', 0)} rumah")
print(f"Persentase Coverage           : {(statistik.get('Terjangkau', 0) / 300) * 100:.2f}%\n")

# ==============================================================================
# 4. VISUALISASI WEBGIS INTERAKTIF
# ==============================================================================
# Inisialisasi Peta (Center di tengah rata-rata ODP)
center_lat = odp_lat.mean()
center_lon = odp_lon.mean()
peta_telco = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=14,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Tiles &copy; Esri'
)

# Layer 1: Coverage Area Sinyal Wi-Fi (Poligon Biru Transparan)
folium.GeoJson(
    gdf_coverage_gps,
    name="Area Coverage Jaringan (400m)",
    style_function=lambda x: {'fillColor': '#2ecc71', 'color': '#27ae60', 'weight': 2, 'fillOpacity': 0.3}
).add_to(peta_telco)

# Layer 2: Pemetaan Status Rumah
for idx, row in gdf_rumah_gps.iterrows():
    if row['Status_Sinyal'] == 'Terjangkau':
        # Titik Hijau untuk yang terjangkau
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=4, color='green', fill=True, popup="Aman"
        ).add_to(peta_telco)
    else:
        # Titik Merah untuk BLIND SPOT
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=4, color='red', fill=True, popup="BLIND SPOT!"
        ).add_to(peta_telco)

# Layer 3: Lokasi Tiang ODP / Router (Marker Hitam)
for idx, row in gdf_odp.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        popup=f"<b>{row['ID_ODP']}</b>",
        icon=folium.Icon(color='black', icon='signal')
    ).add_to(peta_telco)

folium.LayerControl().add_to(peta_telco)

# Tampilkan Peta
peta_telco

Memulai Analisis Jaringan Telekomunikasi...

=== HASIL ANALISIS CAKUPAN JARINGAN (Radius 400m) ===
Total Rumah Terjangkau Sinyal : 9 rumah
Total Rumah BLIND SPOT        : 291 rumah
Persentase Coverage           : 3.00%



/tmp/ipykernel_5046/2517230778.py:50: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  coverage_area = coverage_polygons.unary_union
